# LINet3 Training on ScanNet (Pretrain for SUN Transfer)

**Full training pipeline on ScanNet 20-category with official train/val split.**

Produces a pretrained checkpoint that can be loaded by the SUN HPO notebook
(`LOAD_WEIGHTS = True` in `colab_LiNet3_SUN_hype_tune.ipynb`).

---

## Checklist Before Running:

- [ ] **Enable A100 GPU:** Runtime > Change runtime type > A100
- [ ] **Upload ScanNet dataset to Drive:** `MyDrive/datasets/scannet_pretrain_256.tar.gz`

## 1. Environment Setup & GPU Verification

In [ ]:
# Check GPU availability and specs
import torch
import subprocess

print("=" * 60)
print("GPU VERIFICATION")
print("=" * 60)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

    gpu_name = torch.cuda.get_device_name(0)
    if 'A100' in gpu_name:
        print("\nA100 GPU detected - optimal for training")
    elif 'V100' in gpu_name:
        print("\nV100 GPU detected - good for training (slower than A100)")
    elif 'T4' in gpu_name:
        print("\nT4 GPU detected - will be slower, consider upgrading to A100")
    else:
        print(f"\nGPU: {gpu_name}")
else:
    print("\nNO GPU DETECTED!")
    print("Enable GPU: Runtime -> Change runtime type -> Hardware accelerator: GPU")
    raise RuntimeError("GPU is required for training")

print("\n" + "=" * 60)

In [ ]:
!nvidia-smi

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
import os
from pathlib import Path

# Mount Google Drive
drive.mount('/content/drive')

print("\nGoogle Drive mounted successfully!")
print(f"\nDrive contents:")
!ls -la /content/drive/MyDrive/ | head -20

## 3. Clone Repository to Local Disk (Fast I/O)

In [ ]:
import os
from pathlib import Path

# Configuration
PROJECT_NAME = "Multi-Stream-Neural-Networks"
GITHUB_REPO = "https://github.com/clingergab/Multi-Stream-Neural-Networks.git"
LOCAL_REPO_PATH = f"/content/{PROJECT_NAME}"

print("=" * 60)
print("REPOSITORY SETUP")
print("=" * 60)

os.chdir('/content')

if Path(LOCAL_REPO_PATH).exists() and Path(f"{LOCAL_REPO_PATH}/.git").exists():
    print(f"Repo already exists: {LOCAL_REPO_PATH}")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
else:
    if Path(LOCAL_REPO_PATH).exists():
        !rm -rf {LOCAL_REPO_PATH}
    print(f"Cloning from {GITHUB_REPO}...")
    !git clone {GITHUB_REPO} {LOCAL_REPO_PATH}
    if not Path(LOCAL_REPO_PATH).exists():
        raise RuntimeError(f"Failed to clone repository")
    os.chdir(LOCAL_REPO_PATH)

print(f"\nWorking directory: {os.getcwd()}")
!ls -la {LOCAL_REPO_PATH}
print("\n" + "=" * 60)

## 4. Install Dependencies

In [ ]:
# Install required packages
print("Installing dependencies...")

!pip install -q h5py tqdm matplotlib seaborn ray[tune] kornia thop

# Verify installations
import h5py
import tqdm
import matplotlib
import seaborn
import kornia
import thop

print("All dependencies installed!")
print(f"   h5py: {h5py.__version__}")
print(f"   matplotlib: {matplotlib.__version__}")
print(f"   kornia: {kornia.__version__}")
print(f"   thop: {thop.__version__}")

## 5. Setup Python Path & Imports

In [ ]:
import sys
import os
import json
import math
import shutil
import warnings
import random
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
from collections import Counter

# Remove cached modules
modules_to_reload = [k for k in sys.modules.keys() if k.startswith('src.')]
for module in modules_to_reload:
    del sys.modules[module]

# Add project to Python path
project_root = '/content/Multi-Stream-Neural-Networks'
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Model
from src.models.linear_integration.li_net3 import li_resnet18
from src.models.linear_integration.li_net3.conv import LIConv2d, LIBatchNorm2d
from src.models.linear_integration.li_net3.container import LIReLU
from src.models.linear_integration.li_net3.pooling import LIMaxPool2d, LIAdaptiveAvgPool2d
from src.models.common.model_helpers import load_pretrained_backbone

# Dataset
from src.data_utils.scannet_pretrain_dataset import (
    ScanNetPretrainDataset, _load_norm_stats as scannet_load_norm_stats,
    _load_class_names as scannet_load_class_names, _discover_samples,
)

# Training
from src.training.augmentation_config import AugmentationConfig
from src.training.optimizers import create_stream_optimizer
from src.training.schedulers import setup_scheduler
from src.utils.seed import set_seed

# Visualization (training curves + weight evolution only)
from src.utils.visualization import IntegrationWeightEvolutionVisualizer

# FLOP counting
from thop import profile

print("All imports successful!")

In [ ]:
SEED = 42
DETERMINISTIC = False
set_seed(SEED, deterministic=DETERMINISTIC)
print(f"Seed: {SEED}, Deterministic: {DETERMINISTIC}")

## 6. Copy ScanNet Dataset

Checks if `/dev/shm` has >40 GB free. If yes, extracts to RAM disk for fastest I/O. Otherwise extracts to local SSD (`/content/`).

In [ ]:
DRIVE_SCANNET_TAR = "/content/drive/MyDrive/datasets/scannet_pretrain_256.tar.gz"

print("=" * 60)
print("SCANNET 20-CATEGORY PRETRAIN DATASET SETUP")
print("=" * 60)

# Check /dev/shm free space
shm_stat = shutil.disk_usage("/dev/shm")
shm_free_gb = shm_stat.free / (1024**3)
print(f"/dev/shm free space: {shm_free_gb:.1f} GB")

if shm_free_gb > 40:
    SCANNET_DATA_PATH = "/dev/shm/scannet_pretrain_256"
    extract_target = "/dev/shm"
    print(f"Sufficient RAM \u2014 extracting to RAM disk: {SCANNET_DATA_PATH}")
else:
    SCANNET_DATA_PATH = "/content/scannet_pretrain_256"
    extract_target = "/content"
    print(f"Insufficient RAM ({shm_free_gb:.1f} GB < 40 GB) \u2014 extracting to local SSD: {SCANNET_DATA_PATH}")

if Path(SCANNET_DATA_PATH).exists():
    print(f"\nAlready on local disk: {SCANNET_DATA_PATH}")
    train_dir = Path(f"{SCANNET_DATA_PATH}/train")
    if train_dir.exists():
        train_count = sum(1 for _ in train_dir.rglob("*_rgb.pt"))
        print(f"  Train samples: {train_count}")
elif Path(DRIVE_SCANNET_TAR).exists():
    print(f"\nFound on Drive: {DRIVE_SCANNET_TAR}")
    tar_name = Path(DRIVE_SCANNET_TAR).name
    local_tar = f"{extract_target}/{tar_name}"
    !rsync -ah --info=progress2 {DRIVE_SCANNET_TAR} {local_tar}
    print(f"\nExtracting...")
    !tar -xzf {local_tar} -C {extract_target}/ 2>&1 | grep -v "Ignoring unknown extended header"
    !rm {local_tar}
    train_dir = Path(f"{SCANNET_DATA_PATH}/train")
    train_count = sum(1 for _ in train_dir.rglob("*_rgb.pt"))
    print(f"Extracted. Train samples: {train_count}")
else:
    raise FileNotFoundError(f"Dataset not found at {DRIVE_SCANNET_TAR}")

print(f"\nScanNet dataset ready at: {SCANNET_DATA_PATH}")

## 7. Configuration

All hyperparameters and settings in one place. Fill in from HPO results before running.

In [ ]:
STREAM_LABELS = {0: 'RGB', 1: 'Depth'}

# ======================== SCANNET DATASET ========================
SCANNET_DATASET_CONFIG = {
    'data_root': SCANNET_DATA_PATH,
    'batch_size': 128,
    'num_workers': 4,
    'seed': SEED,
}

SCANNET_AUGMENTATION_CONFIG = AugmentationConfig(
    rgb_aug_prob=0.5,      # TODO: fill from HPO
    rgb_aug_mag=0.5,       # TODO: fill from HPO
    depth_aug_prob=0.5,    # TODO: fill from HPO
    depth_aug_mag=0.5,     # TODO: fill from HPO
)

# ======================== SCANNET MODEL ========================
SCANNET_MODEL_CONFIG = {
    'num_classes': 20,
    'stream_input_channels': [3, 1],
    'width_multiplier': 0.75,
    'dropout_p': 0.25,     # TODO: fill from HPO
    'device': 'cuda',
    'use_amp': True,
}

# ======================== SCANNET OPTIMIZER ========================
SCANNET_OPTIMIZER_CONFIG = {
    'stream_lrs': [1e-3, 1e-3],           # TODO: fill from HPO
    'stream_weight_decays': [5e-5, 5e-5],  # TODO: fill from HPO
    'shared_lr': 1e-3,                     # TODO: fill from HPO
    'integration_weight_decay': 5e-5,      # TODO: fill from HPO
}

SCANNET_SCHEDULER_CONFIG = {
    'scheduler_type': 'cosine',
    't_max': 110,
    'eta_min': [1e-6, 1e-6, 1e-6, 1e-6],  # TODO: fill from HPO
    'warmup_epochs': 5,
    'warmup_start_factor': 0.2,
}

# ======================== SCANNET TRAINING ========================
SCANNET_TRAIN_CONFIG = {
    'epochs': 115,
    'grad_clip_norm': 1.5,             # TODO: fill from HPO
    'early_stopping': True,
    'patience': 15,
    'monitor': 'val_mca',
    'label_smoothing': 0.05,           # TODO: fill from HPO
    'modality_dropout': True,
    'modality_dropout_start': 0,
    'modality_dropout_ramp': 20,
    'modality_dropout_rate': 0.05,
    'stream_monitoring': True,
    'gradient_monitoring': True,
    'gradient_log_freq': 0,
    'track_integration_weights': True,
    'integration_snapshot_freq': 10,
}

# Print summary
print("ScanNet Configuration:")
print(f"  Classes: {SCANNET_MODEL_CONFIG['num_classes']}")
print(f"  Epochs: {SCANNET_TRAIN_CONFIG['epochs']}")
print(f"  Batch size: {SCANNET_DATASET_CONFIG['batch_size']}")
print(f"  Width multiplier: {SCANNET_MODEL_CONFIG['width_multiplier']}")

## 8. Load Dataset

In [ ]:
print("=" * 60)
print("LOADING SCANNET DATASET")
print("=" * 60)

data_root = SCANNET_DATASET_CONFIG['data_root']

# Load dataset metadata
scannet_norm_stats = scannet_load_norm_stats(data_root)
scannet_class_names = scannet_load_class_names(data_root)
scannet_num_classes = len(scannet_class_names)

assert scannet_num_classes == SCANNET_MODEL_CONFIG['num_classes'], \
    f"Expected {SCANNET_MODEL_CONFIG['num_classes']} classes, found {scannet_num_classes}"

# Discover samples
train_samples = _discover_samples(os.path.join(data_root, 'train'), scannet_class_names)
val_samples = _discover_samples(os.path.join(data_root, 'val'), scannet_class_names)

print(f"Classes: {scannet_num_classes} ({scannet_class_names[:3]}...)")
print(f"Train samples: {len(train_samples)}")
print(f"Val samples:   {len(val_samples)}")

# Build datasets
g = torch.Generator().manual_seed(SEED)

train_dataset = ScanNetPretrainDataset(
    data_root=data_root,
    split='train',
    samples=train_samples,
    class_names=scannet_class_names,
    norm_stats=scannet_norm_stats,
    normalize=False,  # GPU will normalize after augmentation
    **SCANNET_AUGMENTATION_CONFIG.to_dict(),
)

val_dataset = ScanNetPretrainDataset(
    data_root=data_root,
    split='val',
    samples=val_samples,
    class_names=scannet_class_names,
    norm_stats=scannet_norm_stats,
    normalize=False,
)

# Stratified sampling for training (class imbalance)
all_labels = train_dataset.labels
label_counts = Counter(all_labels)
n_samples = len(all_labels)
class_weights_map = {label: n_samples / count for label, count in label_counts.items()}
sample_weights = torch.tensor(
    [class_weights_map[label] for label in all_labels], dtype=torch.float32
)

train_sampler = torch.utils.data.WeightedRandomSampler(
    weights=sample_weights,
    num_samples=n_samples,
    replacement=True,
    generator=g,
)

def worker_init_fn(worker_id):
    worker_seed = SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)

scannet_train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=SCANNET_DATASET_CONFIG['batch_size'],
    shuffle=False,
    sampler=train_sampler,
    num_workers=SCANNET_DATASET_CONFIG['num_workers'],
    prefetch_factor=2,
    persistent_workers=True,
    pin_memory=True,
    worker_init_fn=worker_init_fn,
)

scannet_val_loader = torch.utils.data.DataLoader(
    val_dataset,
    batch_size=SCANNET_DATASET_CONFIG['batch_size'],
    shuffle=False,
    num_workers=SCANNET_DATASET_CONFIG['num_workers'],
    prefetch_factor=2,
    persistent_workers=False,
    pin_memory=True,
    worker_init_fn=worker_init_fn,
)

print(f"\nDataloaders created:")
print(f"  Train: {len(scannet_train_loader.dataset)} samples ({len(scannet_train_loader)} batches)")
print(f"  Val:   {len(scannet_val_loader.dataset)} samples ({len(scannet_val_loader)} batches)")
print("=" * 60)

## 9. Create Model

In [ ]:
print("=" * 60)
print("MODEL CREATION")
print("=" * 60)

# Checkpoint directory on Drive (persistent)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
checkpoint_dir = f"/content/drive/MyDrive/linet_checkpoints/scannet_pretrain_{timestamp}"
Path(checkpoint_dir).mkdir(parents=True, exist_ok=True)

SCANNET_TRAIN_CONFIG['save_path'] = f"{checkpoint_dir}/best_model.pt"
SCANNET_TRAIN_CONFIG['integration_snapshot_path'] = f"{checkpoint_dir}/integration_snapshots"
os.makedirs(SCANNET_TRAIN_CONFIG['integration_snapshot_path'], exist_ok=True)

model = li_resnet18(
    num_classes=SCANNET_MODEL_CONFIG['num_classes'],
    stream_input_channels=SCANNET_MODEL_CONFIG['stream_input_channels'],
    width_multiplier=SCANNET_MODEL_CONFIG['width_multiplier'],
    dropout_p=SCANNET_MODEL_CONFIG['dropout_p'],
    device=SCANNET_MODEL_CONFIG['device'],
    use_amp=SCANNET_MODEL_CONFIG['use_amp'],
)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

# GFLOPs calculation with custom handlers for LI modules
def _liconv2d_flops(module, input, output):
    stream_outs, integrated_out = output
    total = 0
    for w, s_out in zip(module.stream_weights, stream_outs):
        batch, out_c, out_h, out_w = s_out.shape
        kernel_ops = w.shape[1] * w.shape[2] * w.shape[3]
        total += batch * out_c * out_h * out_w * kernel_ops
    if module.integrated_weight.shape[1] > 0:
        batch, out_c, out_h, out_w = integrated_out.shape
        kernel_ops = module.integrated_weight.shape[1]
        total += batch * out_c * out_h * out_w * kernel_ops
    for iw in module.integration_from_streams:
        batch, out_c, out_h, out_w = integrated_out.shape
        kernel_ops = iw.shape[1]
        total += batch * out_c * out_h * out_w * kernel_ops
    module.total_ops += torch.DoubleTensor([total])

def _libn_flops(module, input, output):
    stream_outs, integrated_out = output
    total = sum(s.numel() for s in stream_outs) + integrated_out.numel()
    module.total_ops += torch.DoubleTensor([total * 4])

def _lirelu_flops(module, input, output):
    stream_outs, integrated_out = output
    module.total_ops += torch.DoubleTensor([sum(s.numel() for s in stream_outs) + integrated_out.numel()])

def _lipool_flops(module, input, output):
    stream_outs, integrated_out = output
    module.total_ops += torch.DoubleTensor([sum(s.numel() for s in stream_outs) + integrated_out.numel()])

custom_ops = {
    LIConv2d: _liconv2d_flops,
    LIBatchNorm2d: _libn_flops,
    LIReLU: _lirelu_flops,
    LIMaxPool2d: _lipool_flops,
    LIAdaptiveAvgPool2d: _lipool_flops,
}

dummy_streams = [torch.randn(1, ch, 224, 224).to(SCANNET_MODEL_CONFIG['device'])
                 for ch in SCANNET_MODEL_CONFIG['stream_input_channels']]
li_flops, _ = profile(model, inputs=(dummy_streams,), custom_ops=custom_ops, verbose=False)
del dummy_streams

print(f"\nLINet3-ResNet18 created")
print(f"  Total parameters: {total_params:,}")
print(f"  GFLOPs: {li_flops / 1e9:.3f}")
print(f"  Model size: {total_params * 4 / 1024**2:.2f} MB (FP32)")
print(f"  Streams: {STREAM_LABELS}")
print(f"  AMP: {SCANNET_MODEL_CONFIG['use_amp']}")
print(f"\nCheckpoint dir: {checkpoint_dir}")
print("=" * 60)

## 10. Compile Model (Optimizer + Scheduler)

In [ ]:
print("=" * 60)
print("MODEL COMPILATION")
print("=" * 60)

# Create optimizer
optimizer = create_stream_optimizer(
    model,
    optimizer_type='adamw',
    stream_lrs=SCANNET_OPTIMIZER_CONFIG['stream_lrs'],
    stream_weight_decays=SCANNET_OPTIMIZER_CONFIG['stream_weight_decays'],
    shared_lr=SCANNET_OPTIMIZER_CONFIG['shared_lr'],
    integration_weight_decay=SCANNET_OPTIMIZER_CONFIG['integration_weight_decay'],
)

print(f"Optimizer: {optimizer.__class__.__name__}")
for i, group in enumerate(optimizer.param_groups):
    num_params = sum(p.numel() for p in group['params'])
    print(f"  Group {i+1}: lr={group['lr']:.2e}, wd={group['weight_decay']:.2e}, params={num_params:,}")

# Create scheduler
scheduler = setup_scheduler(
    optimizer,
    scheduler_type=SCANNET_SCHEDULER_CONFIG['scheduler_type'],
    eta_min=SCANNET_SCHEDULER_CONFIG['eta_min'],
    t_max=SCANNET_SCHEDULER_CONFIG['t_max'],
    train_loader_len=len(scannet_train_loader),
    warmup_epochs=SCANNET_SCHEDULER_CONFIG['warmup_epochs'],
    warmup_start_factor=SCANNET_SCHEDULER_CONFIG['warmup_start_factor'],
)

# Compile
model.compile(
    optimizer=optimizer,
    scheduler=scheduler,
    loss='cross_entropy',
    label_smoothing=SCANNET_TRAIN_CONFIG['label_smoothing'],
    gpu_augmentation=True,
    norm_stats=scannet_norm_stats,
    **SCANNET_AUGMENTATION_CONFIG.to_dict(),
)

print("\nModel compiled!")
print("=" * 60)

## 11. Train with Full Diagnostics

All diagnostics enabled: gradient health monitoring, per-stream training loss decomposition, integration weight norm tracking + periodic full snapshots.

In [ ]:
warnings.filterwarnings(
    'ignore',
    message='The epoch parameter in `scheduler.step\\(\\)` was not necessary',
    category=UserWarning
)

print("=" * 60)
print("SCANNET PRETRAINING")
print("=" * 60)

scannet_history = model.fit(
    train_loader=scannet_train_loader,
    val_loader=scannet_val_loader,
    epochs=SCANNET_TRAIN_CONFIG['epochs'],
    verbose=True,
    save_path=SCANNET_TRAIN_CONFIG['save_path'],
    early_stopping=SCANNET_TRAIN_CONFIG['early_stopping'],
    patience=SCANNET_TRAIN_CONFIG['patience'],
    restore_best_weights=True,
    grad_clip_norm=SCANNET_TRAIN_CONFIG['grad_clip_norm'],
    stream_monitoring=SCANNET_TRAIN_CONFIG['stream_monitoring'],
    monitor=SCANNET_TRAIN_CONFIG['monitor'],
    modality_dropout=SCANNET_TRAIN_CONFIG['modality_dropout'],
    modality_dropout_start=SCANNET_TRAIN_CONFIG['modality_dropout_start'],
    modality_dropout_ramp=SCANNET_TRAIN_CONFIG['modality_dropout_ramp'],
    modality_dropout_rate=SCANNET_TRAIN_CONFIG['modality_dropout_rate'],
    gradient_monitoring=SCANNET_TRAIN_CONFIG['gradient_monitoring'],
    gradient_log_freq=SCANNET_TRAIN_CONFIG['gradient_log_freq'],
    track_integration_weights=SCANNET_TRAIN_CONFIG['track_integration_weights'],
    integration_snapshot_path=SCANNET_TRAIN_CONFIG['integration_snapshot_path'],
    integration_snapshot_freq=SCANNET_TRAIN_CONFIG['integration_snapshot_freq'],
)

print("\n" + "=" * 60)
print("SCANNET PRETRAINING COMPLETE!")
print("=" * 60)

## 12. Single-Stream Robustness Evaluation

How much does the model degrade when a stream is missing? Tests full model, RGB-only, and Depth-only on the validation set.

In [ ]:
print("\n" + "=" * 60)
print("SINGLE-STREAM ROBUSTNESS EVALUATION (VAL SET)")
print("=" * 60)
print("\nTesting model performance with missing streams...\n")

# Evaluate with all streams (normal)
print("[1/3] Evaluating with BOTH streams (normal):")
results_both = model.evaluate(scannet_val_loader, stream_monitoring=True)
print(f"      Accuracy: {results_both['accuracy']*100:.2f}%  MCA: {results_both['mean_class_accuracy']*100:.2f}%")

# Evaluate with RGB only (Depth blanked)
print("\n[2/3] Evaluating with RGB ONLY (Depth blanked):")
results_rgb_only = model.evaluate(scannet_val_loader, stream_monitoring=True, blanked_streams={1})
print(f"      Accuracy: {results_rgb_only['accuracy']*100:.2f}%  MCA: {results_rgb_only['mean_class_accuracy']*100:.2f}%")

# Evaluate with Depth only (RGB blanked)
print("\n[3/3] Evaluating with DEPTH ONLY (RGB blanked):")
results_depth_only = model.evaluate(scannet_val_loader, stream_monitoring=True, blanked_streams={0})
print(f"      Accuracy: {results_depth_only['accuracy']*100:.2f}%  MCA: {results_depth_only['mean_class_accuracy']*100:.2f}%")

print("\n" + "=" * 60)
print("ROBUSTNESS SUMMARY")
print("=" * 60)
print(f"\n  Both streams:  Acc={results_both['accuracy']*100:.2f}%  MCA={results_both['mean_class_accuracy']*100:.2f}%")
print(f"  RGB only:      Acc={results_rgb_only['accuracy']*100:.2f}%  MCA={results_rgb_only['mean_class_accuracy']*100:.2f}% (Depth missing)")
print(f"  Depth only:    Acc={results_depth_only['accuracy']*100:.2f}%  MCA={results_depth_only['mean_class_accuracy']*100:.2f}% (RGB missing)")

rgb_degradation = (results_both['accuracy'] - results_rgb_only['accuracy']) * 100
depth_degradation = (results_both['accuracy'] - results_depth_only['accuracy']) * 100

print(f"\n  Degradation when Depth missing: {rgb_degradation:+.2f}%")
print(f"  Degradation when RGB missing:   {depth_degradation:+.2f}%")
print("\n" + "=" * 60)

## 13. Validation Set Evaluation + Pathway Analysis

In [ ]:
print("=" * 60)
print("VALIDATION SET EVALUATION")
print("=" * 60)

# Evaluate on val set
results = model.evaluate(data_loader=scannet_val_loader, stream_monitoring=True)

print(f"\nVal Results:")
print(f"  Loss: {results['loss']:.4f}")
print(f"  Overall Accuracy: {results['accuracy']*100:.2f}%")
print(f"  Mean Class Accuracy: {results['mean_class_accuracy']*100:.2f}%")

print(f"\nStream-Specific Performance:")
for i in range(len(SCANNET_MODEL_CONFIG['stream_input_channels'])):
    other = (i + 1) % 2
    solo_acc = results[f'stream_{other}_blanked_acc']
    print(f"  Stream{i} ({STREAM_LABELS[i]}) Solo Accuracy: {solo_acc*100:.2f}%")
    print(f"  Stream{i} ({STREAM_LABELS[i]}) Contribution: {results[f'stream_{i}_contribution']*100:+.2f}%")

# Pathway analysis
print(f"\n{'='*60}")
print("PATHWAY ANALYSIS")
print(f"{'='*60}")
print(f"\nAnalyzing stream pathways and integrated pathway contributions...")

pathway_analysis = model.analyze_pathways(data_loader=scannet_val_loader)

print(f"\nSamples analyzed: {pathway_analysis['samples_analyzed']}")

# Accuracy
print("\nAccuracy:")
print(f"  Full model:      {pathway_analysis['accuracy']['full_model']*100:.2f}%")
for i in range(len(SCANNET_MODEL_CONFIG['stream_input_channels'])):
    acc = pathway_analysis['accuracy'][f'stream{i}_only']
    contrib = pathway_analysis['accuracy'][f'stream{i}_contribution']
    print(f"  {STREAM_LABELS[i]} only:       {acc*100:.2f}%  (contribution ratio: {contrib:.3f})")

# Loss
print("\nLoss:")
print(f"  Full model:      {pathway_analysis['loss']['full_model']:.4f}")
for i in range(len(SCANNET_MODEL_CONFIG['stream_input_channels'])):
    loss_i = pathway_analysis['loss'][f'stream{i}_only']
    loss_contrib = pathway_analysis['loss'][f'stream{i}_contribution']
    print(f"  {STREAM_LABELS[i]} only:       {loss_i:.4f}  (loss ratio: {loss_contrib:.3f})")

# Feature norms
print("\nFeature Norms (mean +/- std):")
for i in range(len(SCANNET_MODEL_CONFIG['stream_input_channels'])):
    mean = pathway_analysis['feature_norms'][f'stream{i}_mean']
    std = pathway_analysis['feature_norms'][f'stream{i}_std']
    print(f"  {STREAM_LABELS[i]}:        {mean:.4f} +/- {std:.4f}")
int_mean = pathway_analysis['feature_norms']['integrated_mean']
int_std = pathway_analysis['feature_norms']['integrated_std']
print(f"  Integrated:  {int_mean:.4f} +/- {int_std:.4f}")

# Training summary
print(f"\n{'='*60}")
print("TRAINING SUMMARY")
print(f"{'='*60}")
print(f"  Initial train loss: {scannet_history['train_loss'][0]:.4f}")
print(f"  Final train loss:   {scannet_history['train_loss'][-1]:.4f}")
print(f"  Initial train acc:  {scannet_history['train_accuracy'][0]*100:.2f}%")
print(f"  Final train acc:    {scannet_history['train_accuracy'][-1]*100:.2f}%")
print(f"  Val accuracy:       {results['accuracy']*100:.2f}%")
print(f"  Val MCA:            {results['mean_class_accuracy']*100:.2f}%")
print(f"  Total epochs:       {len(scannet_history['train_loss'])}")

print("\n" + "=" * 60)

## 14. Training Curves + Gradient Health + Stream Contribution

Training diagnostics: loss, accuracy, LR schedule, gradient norms, per-stream contribution, gradient health.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Loss curve (train + val)
axes[0, 0].plot(scannet_history['train_loss'], label='Train Loss', linewidth=2)
if 'val_loss' in scannet_history and scannet_history['val_loss']:
    axes[0, 0].plot(scannet_history['val_loss'], label='Val Loss', linewidth=2, linestyle='--')
axes[0, 0].set_xlabel('Epoch', fontsize=12)
axes[0, 0].set_ylabel('Loss', fontsize=12)
axes[0, 0].set_title('Training Loss', fontsize=14, fontweight='bold')
axes[0, 0].legend(fontsize=11)
axes[0, 0].grid(True, alpha=0.3)

# Accuracy curves (train + val + per-stream)
axes[0, 1].plot([acc*100 for acc in scannet_history['train_accuracy']], label='Train Acc', linewidth=2, color='green')
if 'val_accuracy' in scannet_history and scannet_history['val_accuracy']:
    axes[0, 1].plot([acc*100 for acc in scannet_history['val_accuracy']], label='Val Acc', linewidth=2, color='darkgreen', linestyle='--')
if 'train_mca' in scannet_history and scannet_history['train_mca']:
    axes[0, 1].plot([m*100 for m in scannet_history['train_mca']], label='Train MCA', linewidth=2, color='darkorange', linestyle=':')
if 'val_mca' in scannet_history and scannet_history['val_mca']:
    axes[0, 1].plot([m*100 for m in scannet_history['val_mca']], label='Val MCA', linewidth=2, color='red', linestyle=':')
stream_train_colors = ['skyblue', 'lightcoral', 'gold', 'lightgreen', 'plum']
for i in range(len(SCANNET_MODEL_CONFIG['stream_input_channels'])):
    color_idx = i % len(stream_train_colors)
    if f'stream_{i}_train_acc' in scannet_history:
        axes[0, 1].plot([acc*100 for acc in scannet_history[f'stream_{i}_train_acc']],
                    label=f'{STREAM_LABELS[i]} Train', linewidth=1, alpha=0.6, linestyle='--',
                    color=stream_train_colors[color_idx])
axes[0, 1].set_xlabel('Epoch', fontsize=12)
axes[0, 1].set_ylabel('Accuracy (%)', fontsize=12)
axes[0, 1].set_yticks([20, 40, 60, 80, 100])
axes[0, 1].set_title('Accuracy', fontsize=14, fontweight='bold')
axes[0, 1].legend(fontsize=9, loc='lower right')
for y in range(0, 101, 10):
    axes[0, 1].axhline(y=y, color='gray', alpha=0.3, linewidth=0.5)
for y in range(5, 100, 10):
    axes[0, 1].axhline(y=y, color='gray', alpha=0.2, linewidth=0.5)
axes[0, 1].grid(True, axis='x', alpha=0.3)

# LR schedule
sampled_lrs = scannet_history['learning_rates'][::max(1, len(scannet_history['learning_rates'])//100)]
axes[0, 2].plot(sampled_lrs, linewidth=2, color='green', label='Base LR')
lr_colors = ['blue', 'red', 'orange', 'purple', 'brown']
for i in range(len(SCANNET_MODEL_CONFIG['stream_input_channels'])):
    color_idx = i % len(lr_colors)
    if f'stream_{i}_lr' in scannet_history:
        axes[0, 2].plot(scannet_history[f'stream_{i}_lr'], linewidth=1, alpha=0.7, linestyle='--',
                    color=lr_colors[color_idx], label=f'{STREAM_LABELS[i]} LR')
axes[0, 2].set_xlabel('Epoch', fontsize=12)
axes[0, 2].set_ylabel('Learning Rate', fontsize=12)
axes[0, 2].set_title('Learning Rate Schedule', fontsize=14, fontweight='bold')
axes[0, 2].legend(fontsize=9, loc='upper right')
axes[0, 2].grid(True, alpha=0.3)

# Gradient norms
stream_val_colors = ['blue', 'red', 'orange', 'green', 'purple']
if 'gradient_norms' in scannet_history and scannet_history['gradient_norms']:
    grad_epochs = range(len(scannet_history['gradient_norms']))
    for i in range(len(SCANNET_MODEL_CONFIG['stream_input_channels'])):
        key = f'stream_{i}'
        norms = [d.get(key, {}).get('mean', 0) for d in scannet_history['gradient_norms']]
        color = stream_val_colors[i % len(stream_val_colors)]
        axes[1, 0].plot(grad_epochs, norms, label=f'{STREAM_LABELS[i]}', color=color, linewidth=1.5)
    shared_norms = [d.get('shared', {}).get('mean', 0) for d in scannet_history['gradient_norms']]
    axes[1, 0].plot(grad_epochs, shared_norms, label='Shared', color='gray', linewidth=1.5, linestyle='--')
    axes[1, 0].set_yscale('log')
    axes[1, 0].set_xlabel('Epoch', fontsize=12)
    axes[1, 0].set_ylabel('Gradient Norm (pre-clip, log)', fontsize=12)
    axes[1, 0].set_title('Per-Stream Gradient Norms (mean)', fontsize=14, fontweight='bold')
    axes[1, 0].legend(fontsize=9)
    axes[1, 0].grid(True, alpha=0.3)
else:
    axes[1, 0].text(0.5, 0.5, 'No gradient data', ha='center', va='center',
                    transform=axes[1, 0].transAxes, fontsize=12)
    axes[1, 0].set_title('Per-Stream Gradient Norms', fontsize=14, fontweight='bold')

# Per-stream contribution
contrib_keys = [f'stream_{i}_train_acc' for i in range(len(SCANNET_MODEL_CONFIG['stream_input_channels']))]
if contrib_keys[0] in scannet_history:
    n_streams = len(SCANNET_MODEL_CONFIG['stream_input_channels'])
    baseline_vals = scannet_history['train_accuracy']
    for i in range(n_streams):
        color = stream_val_colors[i % len(stream_val_colors)]
        other = (i + 1) % n_streams if n_streams == 2 else i
        other_vals = scannet_history[f'stream_{other}_train_acc']
        contrib = []
        epochs_eval = []
        for e, (other_acc, base) in enumerate(zip(other_vals, baseline_vals)):
            if not math.isnan(other_acc):
                contrib.append((base - other_acc) * 100)
                epochs_eval.append(e)
        axes[1, 1].plot(epochs_eval, contrib,
                       label=f'{STREAM_LABELS[i]}', color=color, linewidth=1.5, marker='o', markersize=3)
    axes[1, 1].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    axes[1, 1].set_xlabel('Epoch', fontsize=12)
    axes[1, 1].set_ylabel('Contribution (pp)', fontsize=12)
    axes[1, 1].set_title('Per-Stream Contribution\n(Baseline - Acc w/o Stream)', fontsize=14, fontweight='bold')
    axes[1, 1].legend(fontsize=9)
    axes[1, 1].grid(True, alpha=0.3)
else:
    axes[1, 1].text(0.5, 0.5, 'No stream data', ha='center', va='center',
                    transform=axes[1, 1].transAxes, fontsize=12)
    axes[1, 1].set_title('Per-Stream Contribution', fontsize=14, fontweight='bold')

# Gradient health
if 'gradient_health' in scannet_history and scannet_history['gradient_health']:
    axes[1, 2].axis('off')
    health_text = "Gradient Health Summary:\n\n"
    status_counts = {}
    for h in scannet_history['gradient_health']:
        status = h.get('status', 'unknown') if isinstance(h, dict) else str(h)
        status_counts[status] = status_counts.get(status, 0) + 1
    for status, count in sorted(status_counts.items(), key=lambda x: -x[1]):
        health_text += f"  {status}: {count} epochs\n"
    axes[1, 2].text(0.1, 0.9, health_text, transform=axes[1, 2].transAxes,
                    fontsize=10, verticalalignment='top', fontfamily='monospace')
    axes[1, 2].set_title('Gradient Health', fontsize=14, fontweight='bold')
else:
    axes[1, 2].text(0.5, 0.5, 'No gradient health data', ha='center', va='center',
                    transform=axes[1, 2].transAxes, fontsize=12)
    axes[1, 2].set_title('Gradient Health', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(f"{checkpoint_dir}/training_diagnostics.pdf", dpi=150, bbox_inches='tight')
plt.show()

print(f"ScanNet training diagnostics saved to: {checkpoint_dir}/training_diagnostics.pdf")

## 15. Integration Weight Evolution During Training

In [ ]:
evo_viz = IntegrationWeightEvolutionVisualizer(stream_labels=STREAM_LABELS)

# Stream backbone weight norm evolution
if 'stream_weight_norms' in scannet_history:
    evo_viz.plot_stream_weight_norms(scannet_history, save_path=f"{checkpoint_dir}/stream_weight_evolution.pdf")
    print("Stream weight norm evolution saved.")
else:
    print("No stream weight norm data found.")

# Integration weight norm evolution
if 'integration_weight_norms' in scannet_history:
    evo_viz.plot_norm_evolution(scannet_history, save_path=f"{checkpoint_dir}/integration_weight_evolution.pdf")
    print("Integration weight norm evolution saved.")
else:
    print("No integration weight norm data found.")

# Full weight snapshots
snapshot_dir = SCANNET_TRAIN_CONFIG.get('integration_snapshot_path')
if snapshot_dir and os.path.isdir(snapshot_dir) and os.listdir(snapshot_dir):
    evo_viz.plot_snapshot_heatmaps(snapshot_dir, save_path=f"{checkpoint_dir}/integration_weight_snapshots.png")
    print("Integration weight snapshot heatmaps saved (full, PNG).")
    for layer_name in ['conv1', 'layer1']:
        evo_viz.plot_snapshot_heatmaps(
            snapshot_dir, layer_filter=layer_name,
            save_path=f"{checkpoint_dir}/integration_weight_snapshots_{layer_name}.pdf"
        )
    print("Integration weight snapshot heatmaps saved (conv1 + layer1, PDF).")
else:
    print("No integration weight snapshots found.")

## 16. Save Results & Model

In [ ]:
print("=" * 60)
print("SAVING RESULTS")
print("=" * 60)

# Save training history as JSON
history_path = f"{checkpoint_dir}/training_history.json"
with open(history_path, 'w') as f:
    # Build pathway analysis dict
    pa_json = {
        'accuracy': {k: float(v) for k, v in pathway_analysis['accuracy'].items()},
        'loss': {k: float(v) for k, v in pathway_analysis['loss'].items()},
        'feature_norms': {k: float(v) for k, v in pathway_analysis['feature_norms'].items()},
        'samples_analyzed': pathway_analysis['samples_analyzed'],
    }

    json_history = {
        'train_loss': [float(x) for x in scannet_history['train_loss']],
        'train_accuracy': [float(x) for x in scannet_history['train_accuracy']],
        'train_mca': [float(x) for x in scannet_history.get('train_mca', [])],
        'val_loss': [float(x) for x in scannet_history.get('val_loss', [])],
        'val_accuracy': [float(x) for x in scannet_history.get('val_accuracy', [])],
        'val_mca': [float(x) for x in scannet_history.get('val_mca', [])],
        'learning_rates': [float(x) for x in scannet_history['learning_rates']],
        'model_config': SCANNET_MODEL_CONFIG,
        'dataset_config': {k: str(v) if not isinstance(v, (int, float, bool, type(None))) else v
                           for k, v in SCANNET_DATASET_CONFIG.items()},
        'optimizer_config': SCANNET_OPTIMIZER_CONFIG,
        'scheduler_config': SCANNET_SCHEDULER_CONFIG,
        'training_config': {k: str(v) if not isinstance(v, (int, float, bool, type(None))) else v
                            for k, v in SCANNET_TRAIN_CONFIG.items()},
        'augmentation_config': SCANNET_AUGMENTATION_CONFIG.to_dict(),
        'val_results': {
            'loss': float(results['loss']),
            'accuracy': float(results['accuracy']),
            'mean_class_accuracy': float(results.get('mean_class_accuracy', 0)),
        },
        'pathway_analysis': pa_json,
    }

    json.dump(json_history, f, indent=2)

print(f"Training history saved: {history_path}")

# Save final model (compatible with load_pretrained_backbone)
final_model_path = f"{checkpoint_dir}/final_model.pt"
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': model.optimizer.state_dict(),
    'scheduler_state_dict': model.scheduler.state_dict() if model.scheduler else None,
    'config': SCANNET_MODEL_CONFIG,
    'history': scannet_history,
    'val_accuracy': results['accuracy'],
    'val_mca': results['mean_class_accuracy'],
}, final_model_path)

print(f"Final model saved: {final_model_path}")

# List saved files
print(f"\nAll results saved to: {checkpoint_dir}")
!ls -lh {checkpoint_dir}

print(f"\n{'='*60}")
print("TO USE THESE WEIGHTS IN SUN HPO:")
print(f"{'='*60}")
print(f"  1. Open colab_LiNet3_SUN_hype_tune.ipynb")
print(f"  2. Set LOAD_WEIGHTS = True")
print(f"  3. Set PRETRAINED_WEIGHTS_PATH = \"{final_model_path}\"")
print(f"{'='*60}")

## 17. Summary

**Saved models:**
- `best_model.pt` - Best model checkpoint (by val MCA)
- `final_model.pt` - Final model with full state dict, optimizer, scheduler, history

**Saved data:**
- `training_history.json` - Full training history, configs, val results, pathway analysis
- `integration_snapshots/` - Periodic full integration weight snapshots

**Saved visualizations:**
- `training_diagnostics.pdf` - 2x3 grid: loss, accuracy, LR, gradient norms, contribution, gradient health
- `stream_weight_evolution.pdf` - Stream backbone weight norms over training
- `integration_weight_evolution.pdf` - Integration weight norms per layer over training
- `integration_weight_snapshots.png` - Full weight heatmaps at snapshot epochs

**Next step:** Load `final_model.pt` in the SUN HPO notebook with `LOAD_WEIGHTS = True`.